This is a vectorized version of the environment. How does it work? Simply put, instead of running a single environment, we're running them into batches (vectorization). At each time step, instead of formulating a single action $a$, we'll define it as a vector $a=[0, .., n_\text{envs}]$ where each entry corresponds to the action to be performed for an environment.

Thus, if you run 4 simulatenous environments, your observations space becomes (4, num_observations). Because we use a step size of 10 and have a total of 9 snesors, each observation will result into a space of (40,9) elements. The core idea is to speed up the inference and training time of the model instead of querying a single environment.

In our implementation, based on what model you decide, you should take as inputs the observations reshaped to your liking, and predict the actions $a$ from your policy.

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import numpy as np
from student_client.student_gym_env_vectorized import create_student_gym_env_vectorized

env = create_student_gym_env_vectorized(
            num_envs=4,
            user_token='keyvanatt'
        )

2026-03-05 16:08:18,897 - student_client.student_gym_env_vectorized - WARNING - No .env file found and no explicit parameters provided. Using default values. For better setup, create a .env file with:
SERVER_URL=http://rlchallenge.orailix.com
USER_TOKEN=student_user
ENV_TYPE=DegradationEnv
MAX_STEPS_PER_EPISODE=1000
AUTO_RESET=True
TIMEOUT=30.0
2026-03-05 16:08:21,353 - httpx - INFO - HTTP Request: GET http://rlchallenge.orailix.com/api/v1/version "HTTP/1.1 200 OK"
2026-03-05 16:08:21,356 - student_client.student_gym_env_vectorized - INFO - Client is up to date (version 0.4)
2026-03-05 16:08:21,462 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/session/create "HTTP/1.1 200 OK"
2026-03-05 16:08:21,464 - student_client.student_gym_env_vectorized - INFO - Created new session: b34522fd-e30c-4fc2-82ae-86db7c769e95
2026-03-05 16:08:22,822 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/vectorized/episodes/create "HTTP/1.1 200 OK"
2026-03-0

In [6]:
print(f"Environment created with {env.num_envs} parallel environments")
print(f"   Episode IDs: {env.episode_ids}")

# Reset all environments
print(f"\n🔄 Resetting all environments...")
observations, infos = env.reset()

print(f"   Observations shape: {observations.shape}")
print(f"   First observation: {observations[0]}")

Environment created with 4 parallel environments
   Episode IDs: ['896f3752-7e3e-46fb-a9b6-a1be803b5911', '47fa6d31-e577-48be-8594-70c52b630494', '416ef639-b2ff-4b19-830f-934d233ba81d', '70f5070d-62fa-4749-a8a4-d0a532ac3b6d']

🔄 Resetting all environments...


2026-03-05 16:09:35,698 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/episode/vectorized_reset "HTTP/1.1 200 OK"
2026-03-05 16:09:35,701 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 0 episode ID changed from 896f3752-7e3e-46fb-a9b6-a1be803b5911 to 56c3de27-f92a-4921-913e-7b3355d60d78
2026-03-05 16:09:35,702 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 1 episode ID changed from 47fa6d31-e577-48be-8594-70c52b630494 to 9000b0ab-9a78-4268-a6e4-963e4fe39ea6
2026-03-05 16:09:35,703 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 2 episode ID changed from 416ef639-b2ff-4b19-830f-934d233ba81d to 4f5b8279-23a5-488d-8051-315e0b6fe39e
2026-03-05 16:09:35,704 - student_client.student_gym_env_vectorized - INFO - 🔄 Environment 3 episode ID changed from 70f5070d-62fa-4749-a8a4-d0a532ac3b6d to 6faeba4e-b7b6-4b57-9517-7679557eea48
2026-03-05 16:09:35,705 - student_client.student_gym_env_vectorized - INFO - All 4 

   Observations shape: (4, 9)
   First observation: [7.9721497e+02 1.9372027e+04 3.3581287e+02 1.1199792e+03 3.7207216e-01
 1.3707922e+06 3.9580532e+03 0.0000000e+00 9.9364119e+00]


## Training / Iterations

Here you can iterate through the vectorized environments. You'll notice that actions are a vector where each entry corresponds to the associated environment in the vector.

In A), we automatically reset the envs that have terminated so you can continue for an indefinite amount of steps. As environments don't have the same length, they stop at different times, this helps you reset terminated episodes on the fly.

Tips:
- The step_size return many observations, should you feed each one-by-one in your model, or the full step_size=10 one? The choice is yours!
- There exists multiple ways of exploring the dataset

In [7]:
for step in range(100):

    # A) Check if any environments terminated
    terminated_envs = env.get_terminated_env_indices()
    

    # Generate random actions for all environments
    actions = np.random.randint(0, 3, size=env.num_envs)

    print(f"\n   Step {step + 1}:")
    print(f"      Actions: {actions}")

    # Take step
    observations, rewards, terminateds, truncateds, infos = env.step(actions)
    print(f"      Rewards: {rewards}")
    print(f"      Terminated: {terminateds}")
    for i, obs in enumerate(observations):
        print(f"      Obs {i}: {obs.shape}")
        if obs.shape != (10, 9) and obs.shape != (1, 9):
            print(f"         ⚠️  Unexpected observation shape for env {i}: {obs}")
    print(f"      Active environments: {env.get_active_count()}/{env.num_envs}")

    # Show filtered info (production mode)
    for i, info in enumerate(infos):
        print(f"      Env {i} info: {info}")

    #break

#env.close()


   Step 1:
      Actions: [1 2 0 0]


2026-03-05 16:09:48,138 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/episode/vectorized_step "HTTP/1.1 200 OK"


      Rewards: [-507.85062  150.       443.2929   476.13293]
      Terminated: [False  True False False]
      Obs 0: (10, 9)
      Obs 1: (1, 9)
      Obs 2: (10, 9)
      Obs 3: (10, 9)
      Active environments: 3/4
      Env 0 info: {'step': 10, 'terminated': False, 'truncated': False}
      Env 1 info: {'step': 1, 'terminated': True, 'truncated': False}
      Env 2 info: {'step': 10, 'terminated': False, 'truncated': False}
      Env 3 info: {'step': 10, 'terminated': False, 'truncated': False}

   Step 2:
      Actions: [2 1 0 2]


2026-03-05 16:09:53,294 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/episode/vectorized_step "HTTP/1.1 200 OK"


      Rewards: [150.       0.     397.1142 150.    ]
      Terminated: [ True  True False  True]
      Obs 0: (1, 9)
      Obs 1: (1, 10)
         ⚠️  Unexpected observation shape for env 1: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
      Obs 2: (10, 9)
      Obs 3: (1, 9)
      Active environments: 1/4
      Env 0 info: {'step': 11, 'terminated': True, 'truncated': False}
      Env 1 info: {'terminated': True, 'truncated': False}
      Env 2 info: {'step': 20, 'terminated': False, 'truncated': False}
      Env 3 info: {'step': 11, 'terminated': True, 'truncated': False}

   Step 3:
      Actions: [1 0 1 1]


...

2026-03-05 16:10:25,560 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/episode/vectorized_step "HTTP/1.1 200 OK"


      Rewards: [0. 0. 0. 0.]
      Terminated: [ True  True  True  True]
      Obs 0: (1, 10)
         ⚠️  Unexpected observation shape for env 0: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
      Obs 1: (1, 10)
         ⚠️  Unexpected observation shape for env 1: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
      Obs 2: (1, 10)
         ⚠️  Unexpected observation shape for env 2: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
      Obs 3: (1, 10)
         ⚠️  Unexpected observation shape for env 3: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
      Active environments: 0/4
      Env 0 info: {'terminated': True, 'truncated': False}
      Env 1 info: {'terminated': True, 'truncated': False}
      Env 2 info: {'terminated': True, 'truncated': False}
      Env 3 info: {'terminated': True, 'truncated': False}

   Step 23:
      Actions: [2 1 2 0]


2026-03-05 16:10:26,691 - httpx - INFO - HTTP Request: POST http://rlchallenge.orailix.com/api/v1/episode/vectorized_step "HTTP/1.1 200 OK"


      Rewards: [0. 0. 0. 0.]
      Terminated: [ True  True  True  True]
      Obs 0: (1, 10)
         ⚠️  Unexpected observation shape for env 0: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
      Obs 1: (1, 10)
         ⚠️  Unexpected observation shape for env 1: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
      Obs 2: (1, 10)
         ⚠️  Unexpected observation shape for env 2: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
      Obs 3: (1, 10)
         ⚠️  Unexpected observation shape for env 3: [[0. 0. 0. 0. 0. 0. 0. 0. 0. 0.]]
      Active environments: 0/4
      Env 0 info: {'terminated': True, 'truncated': False}
      Env 1 info: {'terminated': True, 'truncated': False}
      Env 2 info: {'terminated': True, 'truncated': False}
      Env 3 info: {'terminated': True, 'truncated': False}

   Step 24:
      Actions: [1 2 1 0]


KeyboardInterrupt: 

In [22]:
for obs_env in observations:
    print(obs_env.shape)

(10, 9)
(1, 9)
(1, 9)
(10, 9)


In [24]:
env.close()

2026-03-05 09:57:55,381 - student_client.student_gym_env_vectorized - INFO - Closed student vectorized environment with 4 environments


In [51]:
observations

[array([[7.9119489e+02, 1.9313773e+04, 3.3493365e+02, 1.1183016e+03,
         3.7162682e-01, 1.3504325e+06, 3.9524648e+03, 0.0000000e+00,
         9.1000004e+00]], dtype=float32),
 array([[7.9269122e+02, 1.9191648e+04, 3.3450665e+02, 1.1146975e+03,
         3.7097099e-01, 1.3704700e+06, 3.9508916e+03, 0.0000000e+00,
         9.2736034e+00]], dtype=float32),
 array([[7.9539990e+02, 1.9358600e+04, 3.3558124e+02, 1.1196388e+03,
         3.7197024e-01, 1.3640279e+06, 3.9565410e+03, 0.0000000e+00,
         9.4699183e+00]], dtype=float32),
 array([[7.9565344e+02, 1.9362465e+04, 3.3514899e+02, 1.1180753e+03,
         3.7165123e-01, 1.3695479e+06, 3.9540964e+03, 0.0000000e+00,
         9.7564869e+00]], dtype=float32)]